In [1]:
import pandas as pd
import numpy as np
import lazyqsar
import tl2cgen
import treelite
import joblib
import pickle
import time
import h5py
import copy
import os

In [16]:
PATH_TO_MODELS = "/aloy/home/acomajuncosa/Ersilia/mtb/processed/unidock_docking/models"
PATH_TO_COMPILED_MODELS = "/home/acomajuncosa/Documents/mtb-targeted-protein-degradation/processed/unidock_docking/compiled_models"
pocket = "alphafold2_P9WFT7_model_0_pocket_2"
bin = "bin_05"

# Load the model
model = joblib.load(os.path.join(PATH_TO_MODELS, pocket, bin, "LQ_RF.joblib"))

In [17]:
# Load some SMILES
PATH_TO_MFPS = "../processed/enamine_characterization"
IK_TO_SMILES = pickle.load(open(os.path.join(PATH_TO_MFPS, "IK_TO_SMI.pkl"), "rb"))
SMILES = np.array(list(IK_TO_SMILES.values())[:100])

# Load X CheMeleon
X = np.load(os.path.join(PATH_TO_MFPS, "X_CheMeleon.npz"))['X']

In [18]:
# Load CheMeleon descriptors
with h5py.File(os.path.join("../processed/enamine_REAL_characterization/embeddings/", "enamine_REAL_chemeleon_chunk_0.h5"), "r") as f:
    X, ids = f['X'][:10000], f['ids']

print(f"Number of compounds: {len(X)}")

Number of compounds: 10000


In [19]:
start = time.time()
preds = model.predict_proba(X)
end = time.time()
print(f"Prediction time: {end - start:.2f} seconds")

[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:01,  5.02it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:02,  4.50it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


...

[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:02,  4.90it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:02,  4.96it/s]

Prediction time: 23.83 seconds


In [20]:
model_to_compile = copy.deepcopy(model)
model_to_compile.model.models[0].model_.n_jobs

8

In [21]:
for mm in model_to_compile.model.models:
    mm.model_.n_jobs = 1

In [22]:
model_to_compile.model.models[0].model_.n_jobs

1

In [23]:
start = time.time()
preds = model_to_compile.predict_proba(X)
end = time.time()
print(f"Prediction time: {end - start:.2f} seconds")

[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:00, 18.65it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:00, 18.44it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


...

[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:00, 18.85it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 10it [00:00, 19.32it/s]

Prediction time: 6.08 seconds


In [24]:
class TL2cgenRFWrapper:
    """Minimal sklearn-like wrapper for TL2cgen predictor."""
    def __init__(self, libpath: str, nthread: int = 8):
        self.libpath = libpath
        self.predictor = tl2cgen.Predictor(libpath=libpath, nthread=nthread)

    def predict_proba(self, X):
        # Ensure float32 + contiguous for best speed
        X = np.asarray(X, dtype=np.float32, order="C")
        dmat = tl2cgen.DMatrix(X)
        # For RF classification, set pred_margin=False to get probabilities
        out = self.predictor.predict(dmat, pred_margin=False)
        # TL2cgen may return shape (n_samples,) for binary, or (n_samples, 2).
        out = np.asarray(out)
        if out.ndim == 1:
            p1 = out
            p0 = 1.0 - p1
            return np.column_stack([p0, p1])
        return out  # already (n_samples, n_classes)

In [25]:
for i, mm in enumerate(model_to_compile.model.models):

    print(mm.model_)
    tl_model = treelite.sklearn.import_model(mm.model_)
    path_to_output = os.path.join(PATH_TO_COMPILED_MODELS, pocket, bin)
    os.makedirs(path_to_output, exist_ok=True)
    libpath = os.path.join(path_to_output, "model.so")  # .dylib on mac, .dll on windows
    tl2cgen.export_lib(tl_model, toolchain="gcc", libpath=libpath, params={"parallel_comp": 8}, options=["-O3"])
    model_to_compile.model.models[i].model_ = TL2cgenRFWrapper(libpath, nthread=8)

RandomForestClassifier(class_weight='balanced_subsample', criterion='entropy',
                       max_features=0.24484242524861066, max_leaf_nodes=1156,
                       n_estimators=501, n_jobs=1, random_state=42)
[16:58:11] /project/src/compiler/ast/split.cc:35: Parallel compilation enabled; member trees will be divided into 8 translation units.


[16:58:11] /tmp/tmpfk5q71dd/libbuild/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.1.2
The model checkpoint was generated from Treelite version 4.4.1


RandomForestClassifier(class_weight='balanced_subsample', criterion='entropy',
                       max_features=0.24484242524861066, max_leaf_nodes=1156,
                       n_estimators=501, n_jobs=1, random_state=42)
[16:58:29] /project/src/compiler/ast/split.cc:35: Parallel compilation enabled; member trees will be divided into 8 translation units.


[16:58:29] /tmp/tmpfk5q71dd/libbuild/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.1.2
The model checkpoint was generated from Treelite version 4.4.1


RandomForestClassifier(class_weight='balanced_subsample', criterion='entropy',
                       max_features=0.4253823424614421, max_leaf_nodes=3773,
                       n_estimators=476, n_jobs=1, random_state=42)
[16:58:47] /project/src/compiler/ast/split.cc:35: Parallel compilation enabled; member trees will be divided into 8 translation units.


...

[17:00:25] /tmp/tmpfk5q71dd/libbuild/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.1.2
The model checkpoint was generated from Treelite version 4.4.1


RandomForestClassifier(class_weight='balanced_subsample', criterion='entropy',
                       max_features=0.24484242524861066, max_leaf_nodes=1156,
                       n_estimators=501, n_jobs=1, random_state=42)


[17:00:42] /tmp/tmpfk5q71dd/libbuild/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.1.2
The model checkpoint was generated from Treelite version 4.4.1


[17:00:42] /project/src/compiler/ast/split.cc:35: Parallel compilation enabled; member trees will be divided into 8 translation units.
RandomForestClassifier(class_weight='balanced_subsample', criterion='entropy',
                       max_features=0.4253823424614421, max_leaf_nodes=3773,
                       n_estimators=476, n_jobs=1, random_state=42)
[17:00:59] /project/src/compiler/ast/split.cc:35: Parallel compilation enabled; member trees will be divided into 8 translation units.


[17:00:59] /tmp/tmpfk5q71dd/libbuild/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.1.2
The model checkpoint was generated from Treelite version 4.4.1


In [ ]:
start = time.time()
preds = model_to_compile.predict_proba(X)
end = time.time()
print(f"Prediction time: {end - start:.2f} seconds")